# PFE ML — Phase E: Probability Calibration Of The Tuned HGB Winner

Phases A-D established that the tuned HGB ranks companies well: AP 0.30 / AUC 0.88 on 2023. Phase F confirmed that ranking quality holds across sectors, age cohorts, and major legal forms. Phase E answers a different question, important specifically because the **Angular frontend exposes `probabilite_cessation` as a numerical probability to users**:

**Phase E question:** when the model outputs *p = 0.7*, do roughly 70% of those companies actually cease? Or is the model well-ranked but mis-scaled?

Ranking quality (AUC, AP) and calibration are independent dimensions. A high-AUC model can still be badly calibrated — for instance, a `class_weight='balanced'` HGB systematically over-estimates the minority class probability because the loss function up-weights positives. Whether this matters depends on how the score is consumed downstream:
- If the product just sorts companies by risk → ranking is enough, calibration is irrelevant.
- If the product displays *"this company has a 73% chance of ceasing in the next 12 months"* → calibration is essential, otherwise the displayed number is meaningless.

## Methodology

- **Model**: HGB with Phase B tuned hyperparameters, fitted on 2017-2022, evaluated on 2023 (same canonical setup as Phases D and F).
- **Metrics**:
  - **Reliability diagram** — predicted probability vs observed positive rate, binned. The 45° line is perfect calibration.
  - **Brier score** — mean squared error between predicted probability and true label. Lower is better. Decomposes into reliability (calibration error) + resolution (discrimination power) + uncertainty (base-rate-only error).
  - **Expected Calibration Error (ECE)** — average |predicted − observed| weighted by bin size. The headline calibration scalar.
- **Calibration methods compared**:
  - **Uncalibrated** (raw HGB output) — baseline.
  - **Platt (sigmoid) recalibration** — fits a 1D logistic regression on the held-out predictions. Best when miscalibration is monotone.
  - **Isotonic recalibration** — non-parametric, fits any monotone mapping. Better when the miscalibration is non-linear, but needs more data.

**Train / fit / evaluate split for calibration:** to avoid in-sample optimism, we fit the model on 2017-2022, recalibrate the probabilities using a held-out *calibration year* (2022, the most recent training year — same approach as Phase B's walk-forward), and evaluate calibration on 2023.

## What this notebook produces

Under `ml-artifacts/calibration_phase_e/`:
- `reliability_uncalibrated.png`, `reliability_platt.png`, `reliability_isotonic.png` — reliability diagrams.
- `calibration_metrics.csv` — Brier / ECE / log-loss for each of the three variants.
- `calibration_summary.md` — thesis-ready paragraph + recommendation.

## 1. Runtime And Constants

**CPU runtime.** Total ~10-12 min: ~8 min for the model fit, the rest is calibration math (fast) and plotting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
START_YEAR = 2017
CALIB_YEAR = 2022       # held-out for fitting recalibrators
TEST_YEAR = 2023        # final evaluation year (Phase C canonical)
N_BINS = 15             # reliability-diagram bin count

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'
PHASE_E_DIR = Path(ARTIFACTS_DIR) / 'calibration_phase_e'
PHASE_E_DIR.mkdir(parents=True, exist_ok=True)

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('CALIB_YEAR =', CALIB_YEAR)
print('TEST_YEAR  =', TEST_YEAR)
print('OUT_DIR    =', PHASE_E_DIR)

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B (collabs/pfe_ml_colab_tuning_phase_b.ipynb) first.'
    )
tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Load Data (2M Hash Sample, Years ≤ 2023) And Build Three Slices

- **Train** = years `<= CALIB_YEAR - 1` (2017-2021). Fits the model.
- **Calib** = year `CALIB_YEAR` (2022). Held out from model fit; used to fit Platt/Isotonic recalibrators.
- **Test**  = year `TEST_YEAR` (2023). Held out from everything; final evaluation.

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [
    f'l."{TARGET}" IS NOT NULL',
    f'f.prediction_year >= {START_YEAR}',
    f'f.prediction_year <= {TEST_YEAR}',
]
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows (≤ {TEST_YEAR}): {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Years: {sorted(df["prediction_year"].unique())}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]

train_mask = X['prediction_year'] < CALIB_YEAR
calib_mask = X['prediction_year'] == CALIB_YEAR
test_mask  = X['prediction_year'] == TEST_YEAR

X_train = X[train_mask].reset_index(drop=True); y_train = y[train_mask].reset_index(drop=True)
X_calib = X[calib_mask].reset_index(drop=True); y_calib = y[calib_mask].reset_index(drop=True)
X_test  = X[test_mask].reset_index(drop=True);  y_test  = y[test_mask].reset_index(drop=True)

print(f'Train (years < {CALIB_YEAR}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Calib (year = {CALIB_YEAR}):  {len(X_calib):,} rows, {y_calib.sum():,} positives')
print(f'Test  (year = {TEST_YEAR}):   {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Fit The Tuned HGB On Years < 2022, Score Calib And Test

In [ ]:
import importlib, time, app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=int((y_train == 1).sum()),
    train_negative_count=int((y_train == 0).sum()),
    extra_params=tuned_params,
)

print(f'Fitting tuned HGB on years 2017–{CALIB_YEAR-1}...')
start = time.time()
pipeline.fit(X_train, y_train)
print(f'Done in {(time.time()-start)/60:.1f} min')

from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
p_calib = pipeline.predict_proba(X_calib)[:, 1]
p_test_raw = pipeline.predict_proba(X_test)[:, 1]

print(f'\nUncalibrated test ranking: AP={average_precision_score(y_test, p_test_raw):.4f}, '
      f'AUC={roc_auc_score(y_test, p_test_raw):.4f}')

## 5. Fit Two Recalibrators On The 2022 Slice

Both recalibrators learn a 1D mapping `raw_probability → calibrated_probability` on the 2022 hold-out, then we apply that mapping to the 2023 raw predictions.

**Platt** = sigmoid fit (logistic regression on the raw probabilities). Best when miscalibration is a simple monotone S-curve.

**Isotonic** = monotone-step-function fit. Flexible, captures non-monotone-but-monotone-in-rank miscalibration. Risk: can overfit with small calibration sets, but 2022 has ~14K positives, plenty.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

# Platt: fit a logistic regression on the raw probability as a feature
platt = LogisticRegression(C=1e6, solver='lbfgs')
platt.fit(p_calib.reshape(-1, 1), y_calib)
p_test_platt = platt.predict_proba(p_test_raw.reshape(-1, 1))[:, 1]

# Isotonic: non-parametric monotone mapping
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib, y_calib)
p_test_iso = iso.predict(p_test_raw)

print('Recalibrators fitted on 2022 calibration slice.')
print(f'Test predictions ready: raw, Platt, Isotonic.')

## 6. Compute Brier, ECE, Log-Loss On All Three Variants

In [ ]:
def expected_calibration_error(y_true, p_pred, n_bins=N_BINS):
    """Weighted-mean | predicted - observed | across equal-width probability bins."""
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(p_pred, bins) - 1, 0, n_bins - 1)
    ece = 0.0
    total = len(p_pred)
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        pred_mean = p_pred[mask].mean()
        obs_mean = y_true.iloc[mask].mean() if hasattr(y_true, 'iloc') else y_true[mask].mean()
        ece += (n / total) * abs(pred_mean - obs_mean)
    return ece

variants = {
    'uncalibrated': p_test_raw,
    'platt': p_test_platt,
    'isotonic': p_test_iso,
}

rows = []
for name, p in variants.items():
    rows.append({
        'variant': name,
        'brier': brier_score_loss(y_test, p),
        'ece': expected_calibration_error(y_test, p, n_bins=N_BINS),
        'log_loss': log_loss(y_test, np.clip(p, 1e-7, 1 - 1e-7)),
        'mean_predicted': float(np.mean(p)),
        'mean_observed': float(y_test.mean()),
        'ap': average_precision_score(y_test, p),
        'auc': roc_auc_score(y_test, p),
    })
calib_metrics = pd.DataFrame(rows)
print('Calibration metrics on 2023 test set:')
print(calib_metrics.to_string(index=False))

calib_metrics.to_csv(PHASE_E_DIR / 'calibration_metrics.csv', index=False)
print(f'\nSaved: {PHASE_E_DIR / "calibration_metrics.csv"}')

## 7. Reliability Diagrams

Each diagram bins the predicted probability into `N_BINS` equal-width bins and plots `(mean predicted, mean observed)` per bin. The 45° dashed line is perfect calibration. Bars below it = over-confident (predicts higher than reality); above = under-confident.

In [ ]:
import matplotlib.pyplot as plt

def reliability_curve(y_true, p_pred, n_bins=N_BINS):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(p_pred, bins) - 1, 0, n_bins - 1)
    rows = []
    for b in range(n_bins):
        mask = bin_idx == b
        n = int(mask.sum())
        if n == 0:
            continue
        rows.append({
            'bin_low': bins[b],
            'bin_high': bins[b + 1],
            'n': n,
            'mean_pred': float(p_pred[mask].mean()),
            'mean_obs': float(y_true.iloc[mask].mean() if hasattr(y_true, 'iloc') else y_true[mask].mean()),
        })
    return pd.DataFrame(rows)

for name, p in variants.items():
    curve = reliability_curve(y_test, p)
    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=(7, 6.5), sharex=True,
        gridspec_kw={'height_ratios': [3, 1]},
    )
    ax_top.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='perfect calibration')
    ax_top.plot(curve['mean_pred'], curve['mean_obs'], 'o-', color='#0f766e',
                linewidth=2, markersize=7, label=name)
    ax_top.set_ylabel('Mean observed positive rate')
    ax_top.set_title(f'Phase E — Reliability diagram ({name})')
    ax_top.legend()
    ax_top.grid(True, alpha=0.3)
    ax_top.set_xlim(0, 1)
    ax_top.set_ylim(0, 1)
    ax_bot.bar(curve['mean_pred'], curve['n'], width=1 / N_BINS * 0.9,
               color='#94a3b8', alpha=0.7)
    ax_bot.set_yscale('log')
    ax_bot.set_xlabel('Mean predicted probability')
    ax_bot.set_ylabel('rows per bin (log)')
    ax_bot.grid(True, alpha=0.3)
    fig.tight_layout()
    out = PHASE_E_DIR / f'reliability_{name}.png'
    fig.savefig(out, dpi=160)
    plt.show()
    print(f'Saved: {out}')

## 8. Pick The Winner And Write A Summary

In [ ]:
best_by_brier = calib_metrics.loc[calib_metrics['brier'].idxmin(), 'variant']
best_by_ece = calib_metrics.loc[calib_metrics['ece'].idxmin(), 'variant']

raw_row = calib_metrics[calib_metrics['variant'] == 'uncalibrated'].iloc[0]
best_row = calib_metrics[calib_metrics['variant'] == best_by_brier].iloc[0]
ece_drop = raw_row['ece'] - best_row['ece']
brier_drop = raw_row['brier'] - best_row['brier']

decision = ''
if best_by_brier == 'uncalibrated':
    decision = (
        'The uncalibrated HGB output is already the best-calibrated variant — '
        'no recalibration is needed. Ship the raw predict_proba output.'
    )
elif raw_row['ece'] < 0.02:
    decision = (
        f'The uncalibrated ECE is already {raw_row["ece"]:.4f} (< 0.02 threshold for "good enough"). '
        f'Recalibration via {best_by_brier} would reduce ECE by {ece_drop:.4f} (~{100*ece_drop/raw_row["ece"]:.0f}%), '
        f'but the gain is small. Ship the raw output unless the product is highly threshold-sensitive.'
    )
else:
    decision = (
        f'The uncalibrated ECE is {raw_row["ece"]:.4f}, which is materially miscalibrated. '
        f'Recalibrate using **{best_by_brier}** (Brier {best_row["brier"]:.4f}, ECE {best_row["ece"]:.4f} — '
        f'reduction of {brier_drop:.4f} / {ece_drop:.4f} respectively). '
        f'Save the calibrator alongside the model and apply it before exposing probabilites_cessation to users.'
    )

summary_md = (
    '# Phase E — Probability Calibration Summary\n\n'
    f'Model: tuned HGB, trained 2017–{CALIB_YEAR-1}, recalibrated on {CALIB_YEAR}, evaluated on {TEST_YEAR}.\n\n'
    f'## Metrics on 2023 test set\n\n'
    f'{calib_metrics.to_markdown(index=False)}\n\n'
    f'## Verdict\n\n'
    f'- **Best Brier**: {best_by_brier}\n'
    f'- **Best ECE**: {best_by_ece}\n'
    f'- ECE drop from recalibration: {ece_drop:+.4f}\n'
    f'- Brier drop from recalibration: {brier_drop:+.4f}\n\n'
    f'**Recommendation:** {decision}\n\n'
    f'## Note on ranking\n\n'
    f'AP and AUC are unchanged across uncalibrated / Platt / Isotonic. Both recalibrators are monotone, '
    f'so the *order* of companies by score is identical and Phase F\'s segment-robustness results carry over.\n'
)
summary_path = PHASE_E_DIR / 'calibration_summary.md'
summary_path.write_text(summary_md, encoding='utf-8')
print(summary_md)
print(f'\nSaved: {summary_path}')

## 9. How To Read The Reliability Diagram

- **Points on the 45° line** → perfectly calibrated. When the model says `p=0.7`, 70% of those companies actually ceased.
- **Points below the 45° line** → over-confident. Model says `p=0.7` but only 50% ceased. Common with class-weighted training (the loss function pushes predictions toward the minority class).
- **Points above the 45° line** → under-confident. Model says `p=0.3` but 50% ceased. Less common; usually a sign of insufficient signal.
- **The bottom panel (log row count)** shows where most of the population lives. A model can be miscalibrated only in the high-probability bins (rare positives) while being well-calibrated in the dense low-probability bins (most of the population) — which is often fine if the product mostly surfaces the high-confidence end.

## Thesis paragraph

*"Phase E evaluated probability calibration on the 2023 test set, motivated by the product's user-facing presentation of `probabilite_cessation` as a numerical probability rather than a rank. The uncalibrated HGB output achieved Brier X.XXX and ECE Y.YYY. Recalibration on a 2022 hold-out using {chosen method} reduced ECE to Z.ZZZ (Brier W.WWW), restoring the interpretation that p = 0.7 corresponds approximately to a 70% observed positive rate. As both Platt and Isotonic recalibrators are monotone, AP and AUC are unchanged from the Phase B/C results."*

Fill in the actual numbers from sections 6 and 8 when transferring to the thesis.

## What's next

If the recommendation is to ship a recalibrator: save `isotonic.joblib` (or `platt.joblib`) alongside `model.joblib` in `ml-artifacts/`, update `MLRegistry.load()` to load both, and add a `predict_proba_calibrated()` method to the service that chains them. Then **Phase G (latency)** can measure the full inference pipeline including recalibration.